# ### Cell 1 — Install
### Same libraries as Step 02: Unsloth for fast model loading, plus wandb (not used for
### generation, but harmless to have). New Colab notebook, so we start fresh.
### Runtime > Change runtime type > T4 GPU (this needs a GPU to run the SFT model).

In [2]:
!pip install -q unsloth

# ### Cell 2 — Mount Google Drive
### Our SFT adapter (from Step 02) and our dataset (from Step 01) are saved in Drive.
### We mount it here so this notebook can read both.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Cell 3 — Load the SFT model (base + our trained adapter)
### This loads Llama 3.1 8B in 4-bit again, then attaches the LoRA adapter we trained in
### Step 02 — so we get back the exact fine-tuned model, ready to generate real answers.

In [4]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 1024
SFT_ADAPTER_PATH = f"{DRIVE_ROOT}/llama3.1-8b-legal-clause-lora-sft"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,   # loading directly from our saved adapter folder
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

FastLanguageModel.for_inference(model)   # switches model into fast generation mode
print("SFT model loaded and ready for generation.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/llama3.1-8b-legal-clause-lora-sft as a legacy tokenizer.
Unsloth 2026.7.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


SFT model loaded and ready for generation.


In [8]:
import json

with open("data/train.jsonl") as f:
    train_records = [json.loads(line) for line in f]

with open("data/categories.json") as f:
    cat_config = json.load(f)

CATEGORIES = cat_config["categories"]
SYSTEM_PROMPT = cat_config["system_prompt"]

print(f"Loaded {len(train_records)} training clauses.")
print(f"Categories: {CATEGORIES}")

Loaded 1200 training clauses.
Categories: ['Governing Laws', 'Terminations', 'Confidentiality', 'Indemnifications', 'Assignments', 'Notices', 'Severability', 'Counterparts', 'Amendments', 'Survival']


In [9]:
import torch
import random

random.seed(42)

N_CLAUSES_FOR_DPO = 700   # subset of the 1,200 training clauses — keeps generation fast
train_subset = random.sample(train_records, N_CLAUSES_FOR_DPO)

def generate_sample(system_prompt: str, clause_text: str, temperature: float = 0.7) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Clause:\n{clause_text}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=12,          # category names are short — no need for 30
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs.shape[-1]:]   # strip the prompt, keep only new tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

sampled_outputs = []   # list of (clause_text, correct_category, [sampled_answers])
N_SAMPLES_PER_CLAUSE = 2   # was 3 — cuts total generations by a third

for i, rec in enumerate(train_subset):
    clause_text = rec["messages"][1]["content"].replace("Clause:\n", "", 1)
    correct_category = rec["messages"][2]["content"]
    samples = [generate_sample(SYSTEM_PROMPT, clause_text) for _ in range(N_SAMPLES_PER_CLAUSE)]
    sampled_outputs.append((clause_text, correct_category, samples))
    if (i + 1) % 100 == 0:
        print(f"  generated for {i+1}/{len(train_subset)} clauses...")

print(f"\nDone. Generated {N_SAMPLES_PER_CLAUSE} samples each for {len(sampled_outputs)} clauses.")
print("Example:", sampled_outputs[0])


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

  generated for 100/700 clauses...


Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  generated for 200/700 clauses...


Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  generated for 500/700 clauses...


Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  generated for 600/700 clauses...


Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  generated for 700/700 clauses...

Done. Generated 2 samples each for 700 clauses.
Example: ('All notices, requests, demands, consents, instructions or other communications required or permitted hereunder shall be in writing and faxed, mailed, or delivered to each party as follows: (i) if to a Voting Party, at such Voting Party’s address or facsimile number set forth in the Company’s records, or at such other address or facsimile number as a Voting Party shall have furnished the Company in writing; or (ii) if to the Company or CEO, at 6720 N. Scottsdale Road, Suite #390, Scottsdale, Arizona 85253, Attention: Chief Executive Officer, or at such other address as the Company shall have furnished to the Voting Parties in writing. All such notices and communications will be deemed effectively given the earlier of: (I) when received; (II) when delivered personally; (III) one business day after being delivered by facsimile (with written confirmation of successful transmission); (IV) one busi

In [10]:
def is_clean_and_correct(sample: str, correct_category: str) -> bool:
    """A sample counts as 'clean and correct' only if it exactly matches the category
    name and contains nothing else (no rambling, no extra words)."""
    return sample.strip() == correct_category

def pick_rejected(samples: list, correct_category: str) -> tuple:
    """Returns (rejected_text, source) where source is 'model' or 'fallback'."""
    for s in samples:
        if not is_clean_and_correct(s, correct_category) and len(s) > 0:
            return s, "model"
    # fallback: rule-based confusable wrong category
    other_categories = [c for c in CATEGORIES if c != correct_category]
    fallback_category = random.choice(other_categories)
    return fallback_category, "fallback"

dpo_pairs = []
fallback_count = 0
for clause_text, correct_category, samples in sampled_outputs:
    rejected_text, source = pick_rejected(samples, correct_category)
    if source == "fallback":
        fallback_count += 1
    dpo_pairs.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Clause:\n{clause_text}"},
        ],
        "chosen": [{"role": "assistant", "content": correct_category}],
        "rejected": [{"role": "assistant", "content": rejected_text}],
    })

print(f"Built {len(dpo_pairs)} DPO pairs.")
print(f"  from real model mistakes: {len(dpo_pairs) - fallback_count}")
print(f"  from rule-based fallback: {fallback_count}")
print("\nExample pair:")
print(json.dumps(dpo_pairs[0], indent=2))

Built 700 DPO pairs.
  from real model mistakes: 17
  from rule-based fallback: 683

Example pair:
{
  "prompt": [
    {
      "role": "system",
      "content": "You are a legal clause classifier. Read the contract clause and respond with exactly one category from this list: Governing Laws, Terminations, Confidentiality, Indemnifications, Assignments, Notices, Severability, Counterparts, Amendments, Survival. Respond with only the category name and nothing else."
    },
    {
      "role": "user",
      "content": "Clause:\nAll notices, requests, demands, consents, instructions or other communications required or permitted hereunder shall be in writing and faxed, mailed, or delivered to each party as follows: (i) if to a Voting Party, at such Voting Party\u2019s address or facsimile number set forth in the Company\u2019s records, or at such other address or facsimile number as a Voting Party shall have furnished the Company in writing; or (ii) if to the Company or CEO, at 6720 N. Scot

In [11]:
OUT_PATH = "dpo_pairs.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"Saved {len(dpo_pairs)} pairs to {OUT_PATH}")

!cp dpo_pairs.jsonl "{DRIVE_ROOT}/dpo_pairs.jsonl"
print("Copied to Google Drive.")
print("\nStep 03a complete. Ready for Step 03b (actual DPO training).")

Saved 700 pairs to dpo_pairs.jsonl
Copied to Google Drive.

Step 03a complete. Ready for Step 03b (actual DPO training).
